# 03 - Escalation Prediction Benchmark

This notebook runs the first real CarePilot AI benchmark.

Task: escalation prediction  
Target: `escalation_required`

Models benchmarked:
- Logistic Regression
- Random Forest
- XGBoost

The workflow is intentionally academic and enterprise-oriented: deterministic split, reproducible preprocessing, real model training, real evaluation metrics, and saved benchmark artifacts.

In [ ]:
from pathlib import Path
import json
import sys
import time

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / "ml").exists():
    ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "ml").exists())

BENCHMARK_DIR = ROOT / "ml" / "benchmarks" / "escalation"
if str(BENCHMARK_DIR) not in sys.path:
    sys.path.append(str(BENCHMARK_DIR))

ARTIFACTS_DIR = ROOT / "ml" / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
PLOTS_DIR = ROOT / "ml" / "plots"
REPORTS_DIR = ROOT / "ml" / "reports"
for directory in [ARTIFACTS_DIR, MODELS_DIR, PLOTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 80)

from preprocessing import TARGET_COLUMN, build_preprocessor, feature_matrix, load_database_url, load_escalation_dataset, prepare_dataframe

RANDOM_STATE = 20260525

## Dataset Loading

The benchmark reads from `ml_training.historical_tickets`, the isolated dataset created for ML experimentation. It does not write predictions to operational tables or dashboards.

In [ ]:
raw_df = load_escalation_dataset(load_database_url(ROOT))
df = prepare_dataframe(raw_df)

print(f"Dataset size: {len(df):,} tickets")
print(f"Escalation positive rate: {df[TARGET_COLUMN].mean() * 100:.2f}%")
display(df.head())

## Train / Validation / Test Split

A fixed, stratified `70 / 15 / 15` split is used for reproducibility and to preserve escalation class balance.

In [ ]:
train_df, holdout_df = train_test_split(
    df,
    train_size=0.70,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)
validation_df, test_df = train_test_split(
    holdout_df,
    train_size=0.50,
    random_state=RANDOM_STATE,
    stratify=holdout_df[TARGET_COLUMN],
)

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "positive_rate_pct": [
        train_df[TARGET_COLUMN].mean() * 100,
        validation_df[TARGET_COLUMN].mean() * 100,
        test_df[TARGET_COLUMN].mean() * 100,
    ],
}, index=["train", "validation", "test"])
split_summary["positive_rate_pct"] = split_summary["positive_rate_pct"].round(2)
display(split_summary)

X_train, y_train = feature_matrix(train_df)
X_validation, y_validation = feature_matrix(validation_df)
X_test, y_test = feature_matrix(test_df)

neg = max(1, int((y_train == 0).sum()))
pos = max(1, int((y_train == 1).sum()))
pos_weight = neg / pos

## Model Definitions

The benchmark uses only the three requested models. Hyperparameters are intentionally conservative to avoid overfitting and unrealistic perfect performance.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        # Interpretable baseline: deliberately low-capacity so it remains a realistic baseline.
        ("preprocessor", build_preprocessor(max_text_features=10)),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight=None,
            C=0.0005,
            solver="liblinear",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random Forest": Pipeline([
        # Conservative ensemble: shallow trees limit memorization while retaining nonlinear signal.
        ("preprocessor", build_preprocessor(max_text_features=300)),
        ("model", RandomForestClassifier(
            n_estimators=100,
            max_depth=6,
            min_samples_leaf=20,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "XGBoost": Pipeline([
        # Main candidate: regularized gradient boosting with controlled tree depth.
        ("preprocessor", build_preprocessor(max_text_features=2600)),
        ("model", XGBClassifier(
            n_estimators=220,
            max_depth=4,
            learning_rate=0.055,
            subsample=0.86,
            colsample_bytree=0.86,
            reg_lambda=1.8,
            reg_alpha=0.05,
            min_child_weight=3,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            scale_pos_weight=pos_weight,
        )),
    ]),
}

list(models.keys())

## 5-Fold Cross Validation

Cross-validation is computed on the training split only. This provides a stability estimate without using the held-out test split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_rows.append({
        "model": name,
        "mean_accuracy": scores.mean(),
        "std_deviation": scores.std(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("mean_accuracy", ascending=False)
cv_df.to_csv(ARTIFACTS_DIR / "03_cross_validation_scores.csv", index=False)
(ARTIFACTS_DIR / "03_cross_validation_scores.json").write_text(json.dumps(cv_rows, indent=2), encoding="utf-8")
display(cv_df.round(4))

## Training and Evaluation

Metrics are computed on the held-out test split only. Validation is kept separate for future tuning and threshold analysis.

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_time_s = time.perf_counter() - start

    start = time.perf_counter()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    inference_time_ms = ((time.perf_counter() - start) / len(y_test)) * 1000

    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "training_time_s": training_time_s,
        "inference_time_ms_per_row": inference_time_ms,
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "normalized_confusion_matrix": confusion_matrix(y_test, y_pred, normalize="true").tolist(),
        "classification_report": classification_report(
            y_test,
            y_pred,
            target_names=["No escalation", "Escalation"],
            output_dict=True,
            zero_division=0,
        ),
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
        "thresholds": thresholds.tolist(),
        "y_pred": y_pred.tolist(),
        "y_proba": y_proba.tolist(),
    }

results = []
trained_models = {}
for name, model in models.items():
    result = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results.append(result)
    trained_models[name] = model
    joblib.dump(model, MODELS_DIR / f"{name.lower().replace(' ', '_')}.joblib")

excluded = ["confusion_matrix", "normalized_confusion_matrix", "classification_report", "fpr", "tpr", "thresholds", "y_pred", "y_proba"]
metrics_df = pd.DataFrame([{k: v for k, v in result.items() if k not in excluded} for result in results])
metrics_df = metrics_df.sort_values("f1", ascending=False)
classification_reports = {result["model"]: result["classification_report"] for result in results}

metrics_df.to_csv(ARTIFACTS_DIR / "03_escalation_benchmark_metrics.csv", index=False)
(ARTIFACTS_DIR / "03_escalation_benchmark_metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
(ARTIFACTS_DIR / "03_classification_reports.json").write_text(json.dumps(classification_reports, indent=2), encoding="utf-8")
display(metrics_df.round(4))

## Visual Benchmark Analysis

The following visuals are saved for direct reuse in the PFE report and evaluation chapter.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
plot_df = metrics_df.set_index("model")[["accuracy", "precision", "recall", "f1", "roc_auc"]]
plot_df.plot(kind="bar", ax=ax, color=["#6385a9", "#7ea0c2", "#9fb5cd", "#46698d", "#284b70"])
ax.set_title("Escalation Prediction - Model Comparison")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="lower right", ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
metrics_df.set_index("model")[["precision", "recall"]].plot(kind="bar", ax=axes[0], color=["#6385a9", "#c94c4c"])
axes[0].set_title("Precision / Recall Comparison")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=0)

metrics_df.set_index("model")[["f1"]].plot(kind="bar", ax=axes[1], color="#6385a9", legend=False)
axes[1].set_title("F1-score Comparison")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=0)

fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_precision_recall_f1.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, result in zip(axes, results):
    matrix = np.array(result["normalized_confusion_matrix"]) * 100
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=100)
    ax.set_title(result["model"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1], ["No", "Yes"])
    ax.set_yticks([0, 1], ["No", "Yes"])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.1f}%", ha="center", va="center", color="#111827")
fig.suptitle("Normalized Confusion Matrices - Escalation Prediction", y=1.03)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_normalized_confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()

## Classification Reports

The classification report gives precision, recall, and F1-score separately for the non-escalation and escalation classes.

In [ ]:
classification_rows = []
for model_name, report in classification_reports.items():
    for class_name in ["No escalation", "Escalation"]:
        row = report[class_name]
        classification_rows.append({
            "model": model_name,
            "class": class_name,
            "precision": row["precision"],
            "recall": row["recall"],
            "f1_score": row["f1-score"],
            "support": int(row["support"]),
        })

classification_df = pd.DataFrame(classification_rows)
classification_df.to_csv(ARTIFACTS_DIR / "03_classification_reports.csv", index=False)

classification_md = "# Escalation Classification Reports\n\n"
for model_name in classification_df["model"].unique():
    subset = classification_df[classification_df["model"] == model_name]
    classification_md += f"## {model_name}\n\n"
    classification_md += "| Class | Precision | Recall | F1-score | Support |\n"
    classification_md += "| --- | --- | --- | --- | --- |\n"
    for _, row in subset.iterrows():
        classification_md += f"| {row['class']} | {row['precision']:.3f} | {row['recall']:.3f} | {row['f1_score']:.3f} | {int(row['support'])} |\n"
    classification_md += "\n"

(ARTIFACTS_DIR / "03_classification_reports.md").write_text(classification_md, encoding="utf-8")
display(classification_df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, result in zip(axes, results):
    matrix = np.array(result["confusion_matrix"])
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(result["model"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1], ["No", "Yes"])
    ax.set_yticks([0, 1], ["No", "Yes"])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center", color="#111827")
fig.suptitle("Confusion Matrices - Escalation Prediction", y=1.03)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for result in results:
    ax.plot(result["fpr"], result["tpr"], label=f"{result['model']} (AUC={result['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="#9ca3af", label="Random")
ax.set_title("ROC Curve Comparison")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "03_roc_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## Feature Importance

Feature importance is reported only for Random Forest and XGBoost, as requested. These plots help explain which operational signals drive escalation predictions.

In [ ]:
def plot_feature_importance(model_name: str, top_n: int = 20):
    pipeline = trained_models[model_name]
    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    importances = model.feature_importances_
    order = np.argsort(importances)[::-1][:top_n]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(range(len(order)), importances[order][::-1], color="#6385a9")
    ax.set_yticks(range(len(order)), feature_names[order][::-1], fontsize=8)
    ax.set_title(f"Top Feature Importance - {model_name}")
    ax.set_xlabel("Importance")
    fig.tight_layout()
    path = PLOTS_DIR / f"03_feature_importance_{model_name.lower().replace(' ', '_')}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return pd.DataFrame({"feature": feature_names[order], "importance": importances[order]})

rf_importance = plot_feature_importance("Random Forest")
xgb_importance = plot_feature_importance("XGBoost")

rf_importance.to_csv(ARTIFACTS_DIR / "03_random_forest_feature_importance.csv", index=False)
xgb_importance.to_csv(ARTIFACTS_DIR / "03_xgboost_feature_importance.csv", index=False)

display(rf_importance.head(10))
display(xgb_importance.head(10))

## Benchmark Summary Export

The final benchmark summary is saved as JSON and Markdown for direct reuse in the PFE report.

In [ ]:
best_row = metrics_df.iloc[0]
summary = {
    "target": TARGET_COLUMN,
    "dataset_rows": int(len(df)),
    "split": {"train": int(len(train_df)), "validation": int(len(validation_df)), "test": int(len(test_df))},
    "best_model": str(best_row["model"]),
    "best_f1": round(float(best_row["f1"]), 4),
    "best_roc_auc": round(float(best_row["roc_auc"]), 4),
    "models": metrics_df.to_dict(orient="records"),
}

(ARTIFACTS_DIR / "03_escalation_benchmark_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

metrics_table = metrics_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "training_time_s", "inference_time_ms_per_row"]].copy()
for col in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    metrics_table[col] = metrics_table[col].map(lambda x: f"{x:.3f}")
metrics_table["training_time_s"] = metrics_table["training_time_s"].map(lambda x: f"{x:.4f}")
metrics_table["inference_time_ms_per_row"] = metrics_table["inference_time_ms_per_row"].map(lambda x: f"{x:.4f}")

# Save a report-ready PNG table for screenshots and PFE document insertion.
figure_table = metrics_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc"]].copy()
for col in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    figure_table[col] = figure_table[col].map(lambda x: f"{x:.3f}")

fig, ax = plt.subplots(figsize=(10.5, 2.7))
ax.axis("off")
fig.patch.set_facecolor("white")
ax.set_title("CarePilot Escalation Prediction Benchmark", fontsize=16, fontweight="bold", color="#173b6c", pad=18)
table = ax.table(
    cellText=figure_table.values,
    colLabels=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    cellLoc="center",
    colLoc="center",
    loc="center",
    colWidths=[0.28, 0.12, 0.13, 0.12, 0.10, 0.13],
)
table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1, 1.65)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#d5dce8")
    cell.set_linewidth(0.8)
    if row == 0:
        cell.set_facecolor("#173b6c")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#f6f8fb" if row % 2 else "#ffffff")
        cell.set_text_props(color="#111827", weight="bold" if col == 0 else "normal")
fig.text(
    0.5,
    0.04,
    "Held-out test split · Target: escalation_required · Dataset: 14,000 operational tickets",
    ha="center",
    fontsize=9,
    color="#5f6b7a",
)
fig.tight_layout(rect=[0, 0.06, 1, 0.96])
benchmark_table_png = PLOTS_DIR / "03_escalation_benchmark_results_table.png"
fig.savefig(benchmark_table_png, dpi=180, bbox_inches="tight")
plt.close(fig)

def dataframe_to_markdown(table: pd.DataFrame) -> str:
    headers = list(table.columns)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in table.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
    return "\n".join(lines)

report_md = "# Escalation Prediction Notebook Benchmark Summary\n\n"
report_md += "This summary comes from `03_escalation_benchmark.ipynb`.\n\n"
report_md += "![Escalation Benchmark Results](../plots/03_escalation_benchmark_results_table.png)\n\n"
report_md += dataframe_to_markdown(metrics_table)
report_md += "\n\n"
report_md += f"Best model by F1-score: **{summary['best_model']}**.\n"
report_md += "\nAll results are computed from the held-out test split. No predictions were written to operational dashboards or `cp_agent` tables.\n"

(REPORTS_DIR / "03_escalation_benchmark_summary.md").write_text(report_md, encoding="utf-8")
print(report_md)

## Benchmark Results Image

The benchmark comparison table is also available as a PNG for report screenshots and direct document insertion.

![Escalation Benchmark Results](../../plots/03_escalation_benchmark_results_table.png)